# Cahya/bert-base-indonesian-NER

source : https://huggingface.co/cahya/bert-base-indonesian-NER

# Setup

In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 101.7 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
from transformers import pipeline
import pandas as pd
import json

# Load Model

In [3]:
pipe = pipeline("token-classification", model="cahya/bert-base-indonesian-NER", aggregation_strategy="simple")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: cahya/bert-base-indonesian-NER
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.pooler.dense.weight     | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [12]:
text = "Desember 1618, laksamana Inggris Thomas Dale mengusir Jan Pieterszoon Coen dari pelabuhan Jayakarta. Coen lari ke Maluku, saat itu pangkalan utama VOC. Kemudian Dale, dibantu Wijayakrama, mengepung benteng VOC."
results = pipe(text)
print(results)

[{'entity_group': 'DAT', 'score': np.float32(0.98645806), 'word': 'desember 1618', 'start': 0, 'end': 13}, {'entity_group': 'NOR', 'score': np.float32(0.6019839), 'word': 'laksamana inggris', 'start': 15, 'end': 32}, {'entity_group': 'PER', 'score': np.float32(0.98464334), 'word': 'thomas dale', 'start': 33, 'end': 44}, {'entity_group': 'PER', 'score': np.float32(0.97510624), 'word': 'jan pieterszoon coen', 'start': 54, 'end': 74}, {'entity_group': 'FAC', 'score': np.float32(0.7606075), 'word': 'pelabuhan', 'start': 80, 'end': 89}, {'entity_group': 'LOC', 'score': np.float32(0.49715668), 'word': 'jayakarta', 'start': 90, 'end': 99}, {'entity_group': 'PER', 'score': np.float32(0.97156006), 'word': 'coen', 'start': 101, 'end': 105}, {'entity_group': 'GPE', 'score': np.float32(0.994218), 'word': 'maluku', 'start': 114, 'end': 120}, {'entity_group': 'LOC', 'score': np.float32(0.32959464), 'word': 'voc', 'start': 147, 'end': 150}, {'entity_group': 'PER', 'score': np.float32(0.9906585), 'wor

In [13]:
for item in results:
  kata = item['word']
  jenis = item['entity_group']
  if item['score'] > 0.8:
    print(f"- {kata} ({jenis}) | score: {item['score']:.2%}")


- desember 1618 (DAT) | score: 98.65%
- thomas dale (PER) | score: 98.46%
- jan pieterszoon coen (PER) | score: 97.51%
- coen (PER) | score: 97.16%
- maluku (GPE) | score: 99.42%
- dale (PER) | score: 99.07%


# Load Data

In [14]:
news_df = pd.read_csv('400_news_2025.csv')
news_df.head()

,summary
0,"BARUS – Bupati Tapteng, Masinton Pasaribu mere..."
1,“There is something just so effortlessly cool ...
2,NUSADAILY.COM – TORAJA UTARA- Sebuah truk yang...
3,NEWS SUMATERA UTARA Bupati Terima Audiensi P...
4,BATAM NEWS Pedagang Ini Sebut 90 % Beras Di ...


In [15]:
PROD_ENTITY_KEYS = [
    'cardinal',
    'date',
    'event',
    'facility',
    'geopolitical',
    'language',
    'law',
    'location',
    'money',
    'ordinal',
    'organization',
    'percentage',
    'person',
    'politicalorganization',
    'product',
    'quantity',
    'religion',
    'time',
    'woart'
]

CAHYA_TO_PROD = {
    'CRD': 'cardinal',
    'DAT': 'date',
    'EVT': 'event',
    'FAC': 'facility',
    'GPE': 'geopolitical',
    'LAN': 'language',
    'LAW': 'law',
    'LOC': 'location',
    'MON': 'money',
    'ORD': 'ordinal',
    'ORG': 'organization',
    'PRC': 'percentage',
    'PER': 'person',
    'NOR': 'politicalorganization',
    'PRD': 'product',
    'QTY': 'quantity',
    'REG': 'religion',
    'TIM': 'time',
    'WOA': 'woart'
}

def build_ners_dict(text, score_threshold=0.8):
    ners = {key: [] for key in PROD_ENTITY_KEYS}
    predictions = pipe(text)

    for pred in predictions:
        short_label = pred['entity_group']
        score = pred['score']

        if score < score_threshold:
            continue

        prod_label = CAHYA_TO_PROD.get(short_label)
        if prod_label is None:
            continue

        entity_text = pred['word'].strip().lower()
        if entity_text and entity_text not in ners[prod_label]:
            ners[prod_label].append(entity_text)

    return ners

def predict_entity(df, text_column='summary', score_threshold=0.8):
    output_df = df[[text_column]].copy()
    output_df['ners'] = output_df[text_column].apply(
        lambda text: json.dumps(build_ners_dict(str(text), score_threshold), ensure_ascii=False)
    )
    return output_df

In [ ]:
results = predict_entity(news_df, text_column='summary', score_threshold=0.8)
results.to_csv('400_news_2025_with_ners.csv', index=False)

,summary,ners
0,"BARUS – Bupati Tapteng, Masinton Pasaribu mere...","{""cardinal"": [], ""date"": [], ""event"": [], ""fac..."
1,“There is something just so effortlessly cool ...,"{""cardinal"": [], ""date"": [], ""event"": [], ""fac..."
2,NUSADAILY.COM – TORAJA UTARA- Sebuah truk yang...,"{""cardinal"": [""4"", ""16""], ""date"": [], ""event"":..."
3,NEWS SUMATERA UTARA Bupati Terima Audiensi P...,"{""cardinal"": [], ""date"": [""rabu ( 26 / 01 / 20..."
4,BATAM NEWS Pedagang Ini Sebut 90 % Beras Di ...,"{""cardinal"": [], ""date"": [], ""event"": [], ""fac..."


In [18]:
results

,summary,ners
0,"BARUS – Bupati Tapteng, Masinton Pasaribu mere...","{""cardinal"": [], ""date"": [], ""event"": [], ""fac..."
1,“There is something just so effortlessly cool ...,"{""cardinal"": [], ""date"": [], ""event"": [], ""fac..."
2,NUSADAILY.COM – TORAJA UTARA- Sebuah truk yang...,"{""cardinal"": [""4"", ""16""], ""date"": [], ""event"":..."
3,NEWS SUMATERA UTARA Bupati Terima Audiensi P...,"{""cardinal"": [], ""date"": [""rabu ( 26 / 01 / 20..."
4,BATAM NEWS Pedagang Ini Sebut 90 % Beras Di ...,"{""cardinal"": [], ""date"": [], ""event"": [], ""fac..."
...,...,...
395,"Jakarta - Kebakaran di Rusun Klender, Malaka ...","{""cardinal"": [], ""date"": [""sabtu ( 12 / 7 / 20..."
396,Jakarta - Jasad seorang laki-laki yang ditemu...,"{""cardinal"": [], ""date"": [""rabu ( 9 / 7 )""], ""..."
397,New York - Piala Dunia Antarklub 2025 dikecam...,"{""cardinal"": [], ""date"": [], ""event"": [""piala ..."
398,"FK-Newcastle, 12 Juli 2025 – Newcastle United ...","{""cardinal"": [], ""date"": [""12 juli 2025"", ""202..."


In [17]:
results.shape

(400, 2)